# Ecobici — 00. Datos crudos y limpieza

Carga de datos y limpieza.

- **Mes principal:** septiembre 2026 (octubre 2026 aún no aparece publicado).
- **Ventana para escenarios:** octubre 2025 – septiembre 2026.
- **Subred:** 5 estaciones del corredor Buenavista–Reforma: 3 en Reforma (014, 027, 018), que se llenan en la mañana, y 2 en Buenavista (271-272, 273-274), que se vacían en la mañana. Así la red tiene estaciones que reciben y estaciones que entregan bicis, que es lo que hace posible redistribuir dentro de la red. 

## 1. Configuración

Definimos las rutas, las URLs de los 12 meses y las estaciones de la subred. 

In [18]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import requests
from pandas.api.types import union_categoricals

pd.set_option("display.max_columns", None)

DIR_TRIPS = Path("data/raw/trips")
DIR_LIMPIO = Path("data/clean")
DIR_TABLAS = Path("outputs/tablas")
RUTA_ESTACIONES = Path("data/raw/station_information.json")

MES_PRINCIPAL = "2026-09"
INICIO, FIN = "2025-09-01", "2026-10-01"    

BASE = "https://ecobici.cdmx.gob.mx/wp-content/uploads"
URLS = {
    "2025-09": f"{BASE}/2025/10/2025-09.csv",        
    "2025-10": f"{BASE}/2025/11/2025-10-1.csv",
    "2025-11": f"{BASE}/2025/12/2025-11.csv",
    "2025-12": f"{BASE}/2026/01/2025-12.csv",
    "2026-01": f"{BASE}/2026/02/2026-01.csv",
    "2026-02": f"{BASE}/2026/03/2026-02.csv",
    "2026-03": f"{BASE}/2026/04/2026-03.csv",
    "2026-04": f"{BASE}/2026/05/2026-04.csv",
    "2026-05": f"{BASE}/2026/06/2026-05.csv",
    "2026-06": f"{BASE}/2026/07/public_data_web_2026-06.csv",
    "2026-07": f"{BASE}/2026/08/public_data_web_2026-07.csv",
    "2026-08": f"{BASE}/2026/09/public_data_web_2026-08_2.csv",
    "2026-09": f"{BASE}/2026/10/public_data_web_2026-09.csv",
}
URL_ESTACIONES = "https://gbfs.mex.lyftbikes.com/gbfs/en/station_information.json"

SUBRED = {
    "014": "Reforma",
    "027": "Reforma",
    "018": "Reforma",
    "271-272": "Buenavista",
    "273-274": "Buenavista",
}

COL_RETIRO = "Ciclo_Estacion_Retiro"
COL_ARRIBO = "Ciclo_EstacionArribo"
FORMATO = "%d/%m/%Y %H:%M:%S"

## 2. Descarga

Los archivos se guardan como `AAAA-MM.csv`, porque en el sitio tienen nombres distintos (`2025-10-1.csv`, `public_data_web_2026-08_2.csv`). Cada descarga se escribe primero en un archivo `.part`, para que un corte no deje un CSV a medias.

In [19]:
def descargar(url, ruta):
    """Descarga url en ruta si el archivo no existe."""
    if ruta.exists():
        return "ya existía"
    ruta.parent.mkdir(parents=True, exist_ok=True)
    temporal = ruta.with_name(ruta.name + ".part")
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(temporal, "wb") as f:
            for bloque in r.iter_content(chunk_size=1 << 20):
                f.write(bloque)
    temporal.rename(ruta)
    return "descargado"


rutas = {mes: DIR_TRIPS / f"{mes}.csv" for mes in URLS}
for mes, url in URLS.items():
    estado = descargar(url, rutas[mes])
    print(f"{mes}  {rutas[mes].stat().st_size / 1e6:7.1f} MB  {estado}")

print("estaciones:", descargar(URL_ESTACIONES, RUTA_ESTACIONES))

2025-09     97.3 MB  descargado
2025-10    107.6 MB  ya existía
2025-11     98.2 MB  ya existía
2025-12     88.5 MB  ya existía
2026-01     93.4 MB  ya existía
2026-02     93.7 MB  ya existía
2026-03    101.3 MB  ya existía
2026-04     97.8 MB  ya existía
2026-05     99.0 MB  ya existía
2026-06     96.1 MB  ya existía
2026-07     96.1 MB  ya existía
2026-08     94.8 MB  ya existía
2026-09     96.4 MB  ya existía
estaciones: ya existía


## 3. Carga de los 12 meses en un solo DataFrame

Leemos los 12 archivos y los juntamos en un solo DataFrame. Todas las reglas de limpieza se aplican una sola vez sobre este DataFrame, así todos los meses se tratan igual.

Para que quepa en memoria, al leer cada archivo:
- Se unen fecha y hora en una sola columna de fecha-hora (`retiro`, `arribo`) y se descartan las columnas de texto originales de fecha y hora.
- Los códigos de estación, bici, género y edad se guardan como `category`. Siguen siendo texto (`014` conserva el cero), pero ocupan mucho menos.
- La columna `archivo` indica de qué CSV viene cada viaje.

No se elimina ninguna fila en este paso.

In [20]:
def cargar_mes(ruta, mes):
    m = pd.read_csv(ruta, dtype=str)
    m["archivo"] = mes
    m["retiro"] = pd.to_datetime(m["Fecha_Retiro"] + " " + m["Hora_Retiro"], format=FORMATO, errors="coerce")
    m["arribo"] = pd.to_datetime(m["Fecha_Arribo"] + " " + m["Hora_Arribo"], format=FORMATO, errors="coerce")
    m = m.drop(columns=["Fecha_Retiro", "Hora_Retiro", "Fecha_Arribo", "Hora_Arribo"])
    for c in ["Genero_Usuario", "Edad_Usuario", "Bici", COL_RETIRO, COL_ARRIBO, "archivo"]:
        m[c] = m[c].astype("category")
    return m

partes, control = [], []
columnas_ref = None
for mes, ruta in rutas.items():
    columnas = list(pd.read_csv(ruta, nrows=0).columns)
    columnas_ref = columnas_ref or columnas
    m = cargar_mes(ruta, mes)
    control.append({"archivo": mes, "viajes": len(m), "mismas_columnas": columnas == columnas_ref,
                    "fechas_no_legibles": int(m[["retiro", "arribo"]].isna().any(axis=1).sum())})
    partes.append(m)
    print(f"{mes} cargado")

# Las categorías deben coincidir entre archivos para poder juntarlos (y retiro/arribo comparten estaciones)
cats_est = union_categoricals([p[c] for p in partes for c in (COL_RETIRO, COL_ARRIBO)]).categories
for p in partes:
    for c in (COL_RETIRO, COL_ARRIBO):
        p[c] = p[c].cat.set_categories(cats_est)
for c in ["Genero_Usuario", "Edad_Usuario", "Bici", "archivo"]:
    cats = union_categoricals([p[c] for p in partes]).categories
    for p in partes:
        p[c] = p[c].cat.set_categories(cats)

df = pd.concat(partes, ignore_index=True)
del partes

estaciones = pd.DataFrame(json.loads(RUTA_ESTACIONES.read_text())["data"]["stations"])

control = pd.DataFrame(control).set_index("archivo")
print(f"\nViajes totales: {len(df):,}   Memoria: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
control

2025-09 cargado
2025-10 cargado
2025-11 cargado
2025-12 cargado
2026-01 cargado
2026-02 cargado
2026-03 cargado
2026-04 cargado
2026-05 cargado
2026-06 cargado
2026-07 cargado
2026-08 cargado
2026-09 cargado

Viajes totales: 20,264,648   Memoria: 0.53 GB


,viajes,mismas_columnas,fechas_no_legibles
archivo,,,
2025-09,1586508,True,0
2025-10,1754353,True,0
2025-11,1600741,True,0
2025-12,1445068,True,0
2026-01,1525302,True,0
2026-02,1528610,True,0
2026-03,1653373,True,0
2026-04,1594967,True,0
2026-05,1615360,True,0


## 4. Vistazo a los datos crudos

Forma general, nulos, rango de fechas y conteos crudos de las 5 estaciones de la subred.

In [21]:
display(df.head())
display(pd.DataFrame({"nulos": df.isna().sum(), "valores_distintos": df.nunique()}))
print("Rango de retiro:", df.retiro.min(), "→", df.retiro.max())
print("Rango de arribo:", df.arribo.min(), "→", df.arribo.max())

info = estaciones.set_index("short_name")
pd.DataFrame({
    "grupo": pd.Series(SUBRED),
    "nombre": info["name"].reindex(list(SUBRED)),
    "docks": info["capacity"].reindex(list(SUBRED)),
    "retiros_12m": df[COL_RETIRO].value_counts().reindex(list(SUBRED), fill_value=0),
    "arribos_12m": df[COL_ARRIBO].value_counts().reindex(list(SUBRED), fill_value=0),
})

,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Ciclo_EstacionArribo,archivo,retiro,arribo
0,F,32,3584745,038,612,2025-09,2025-08-31 23:23:31,2025-09-01 00:00:18
1,M,35,7377708,339,635,2025-09,2025-08-31 23:45:46,2025-09-01 00:00:25
2,F,27,5449988,593,573,2025-09,2025-08-31 23:45:59,2025-09-01 00:00:36
3,M,37,6596803,412,624,2025-09,2025-08-31 23:51:38,2025-09-01 00:00:38
4,M,46,3795499,565,163,2025-09,2025-08-31 23:54:26,2025-09-01 00:00:44


,nulos,valores_distintos
Genero_Usuario,12,4
Edad_Usuario,707,160
Bici,0,7711
Ciclo_Estacion_Retiro,0,682
Ciclo_EstacionArribo,0,682
archivo,0,13
retiro,0,13204824
arribo,0,13218922


Rango de retiro: 2022-08-28 17:20:21 → 2026-09-30 23:56:06
Rango de arribo: 2025-09-01 00:00:18 → 2026-09-30 23:59:57


,grupo,nombre,docks,retiros_12m,arribos_12m
014,Reforma,CE-014 Reforma - Río de Plata,39,89348,112005
027,Reforma,CE-027 Reforma- Havre,43,112092,116740
018,Reforma,CE-018 Reforma - Río Rhin,39,84091,88895
271-272,Buenavista,CE-271-272 Jesús García - Carlos J. Meneses,74,133716,220446
273-274,Buenavista,CE-273-274 Luis Donaldo Colosio - Av. Jesús Ga...,70,61298,80339


## 5. Reglas de limpieza

Se aplican en orden sobre los 12 meses juntos y se registra cuántos viajes elimina cada una. La tabla va tal cual al reporte.

1. **Fecha u hora no legible:** no se puede ubicar el viaje en el tiempo.
2. **Duplicados (misma bici, estaciones y horas):** el mismo viaje registrado más de una vez.
3. **Estación inexistente en el feed:** códigos sin ubicación ni capacidad. Aparecen `1000` y `Temporal 1` a `Temporal 4` (probablemente estaciones temporales para eventos). Son pocos viajes y ninguno toca la subred.
4. **Duración ≤ 0:** arribo antes o igual al retiro.
5. **Duración > 180 min:** bicis perdidas o mal cerradas; un viaje normal dura ~12 min y el percentil 99 está cerca de 50 min.
6. **Misma estación y menos de 3 min:** el usuario sacó la bici y la devolvió sin viajar. No se eliminan viajes cortos entre estaciones distintas: hay estaciones a menos de 200 m (271-272 y 273-274).
7. **Velocidad imposible (> 25 km/h):** distancia en línea recta entre estaciones entre la duración; las bicis eléctricas están limitadas a 25 km/h.
8. **Retiro fuera de la ventana (oct 2025 – sep 2026):** cada archivo se corta por fecha de arribo y trae viajes que empezaron antes.

In [22]:
DUR_MAX_MIN, DUR_MIN_MISMA, VEL_MAX_KMH = 180, 3, 25

def limpiar(d, inicio, fin):
    log = [("Registros originales", 0, len(d))]

    def aplicar(regla, quitar):
        nonlocal d
        n = int(quitar.sum())
        d = d[~quitar]
        log.append((regla, n, len(d)))

    aplicar("Fecha u hora no legible", d["retiro"].isna() | d["arribo"].isna())
    aplicar("Duplicados (misma bici, estaciones y horas)", d.duplicated(subset=["Bici", COL_RETIRO, COL_ARRIBO, "retiro", "arribo"]))

    validas = set(estaciones["short_name"])
    aplicar("Estación inexistente en el feed", ~d[COL_RETIRO].isin(validas) | ~d[COL_ARRIBO].isin(validas))

    d = d.assign(duracion_min=(d["arribo"] - d["retiro"]).dt.total_seconds() / 60)
    aplicar("Duración ≤ 0", d["duracion_min"] <= 0)
    aplicar(f"Duración > {DUR_MAX_MIN} min", d["duracion_min"] > DUR_MAX_MIN)
    aplicar(f"Misma estación y menos de {DUR_MIN_MISMA} min (no hubo viaje)",
            (d["duracion_min"] < DUR_MIN_MISMA) & (d[COL_RETIRO] == d[COL_ARRIBO]))

    coords = estaciones.set_index("short_name")[["lat", "lon"]]
    lat1 = np.radians(d[COL_RETIRO].map(coords["lat"]).astype(float))
    lon1 = np.radians(d[COL_RETIRO].map(coords["lon"]).astype(float))
    lat2 = np.radians(d[COL_ARRIBO].map(coords["lat"]).astype(float))
    lon2 = np.radians(d[COL_ARRIBO].map(coords["lon"]).astype(float))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    d = d.assign(distancia_km=2 * 6371 * np.arcsin(np.sqrt(a)))
    d = d.assign(velocidad_kmh=d["distancia_km"] / (d["duracion_min"] / 60))
    aplicar(f"Velocidad imposible (> {VEL_MAX_KMH} km/h)", d["velocidad_kmh"] > VEL_MAX_KMH)

    aplicar(f"Retiro fuera de la ventana ({inicio} a {fin})", (d["retiro"] < inicio) | (d["retiro"] >= fin))

    log = pd.DataFrame(log, columns=["regla", "eliminados", "restantes"])
    log["pct_eliminado"] = (100 * log["eliminados"] / log.loc[0, "restantes"]).round(3)
    return d.reset_index(drop=True), log


df, log_limpieza = limpiar(df, INICIO, FIN)
log_limpieza

,regla,eliminados,restantes,pct_eliminado
0,Registros originales,0,20264648,0.000
1,Fecha u hora no legible,0,20264648,0.000
2,"Duplicados (misma bici, estaciones y horas)",0,20264648,0.000
3,Estación inexistente en el feed,1946,20262702,0.010
4,Duración ≤ 0,0,20262702,0.000
5,Duración > 180 min,8173,20254529,0.040
6,Misma estación y menos de 3 min (no hubo viaje),34149,20220380,0.169
7,Velocidad imposible (> 25 km/h),2991,20217389,0.015
8,Retiro fuera de la ventana (2025-09-01 a 2026-...,93,20217296,0.000


## 6. Columnas auxiliares

No se elimina ninguna fila: se agregan columnas para filtrar con máscaras.

- `mes`: mes del retiro (para separar el mes objetivo y comparar meses).
- `fecha`, `hora_retiro`, `hora_arribo`, `dia_semana` (0 = lunes).
- `es_habil`: lunes a viernes, sin festivos oficiales ni los días irregulares del 14 y 15 de septiembre (cierre de la zona de Buenavista).
- `en_horario`: retiro entre 05:00 y 23:59.
- `retiro_subred`, `arribo_subred`, `toca_subred`, `grupo_retiro`, `grupo_arribo`.
- `docks_retiro`, `docks_arribo`: capacidad según el feed.

In [23]:
FESTIVOS = pd.to_datetime(["2025-09-16", "2025-11-17", "2025-12-25", "2026-01-01", "2026-02-02",
                           "2026-03-16", "2026-05-01", "2026-09-16"])
                           
DIAS_IRREGULARES = pd.to_datetime(["2026-09-14", "2026-09-15"])

df["mes"] = df["retiro"].dt.strftime("%Y-%m").astype("category")
df["fecha"] = df["retiro"].dt.normalize()
df["hora_retiro"] = df["retiro"].dt.hour.astype("int8")
df["hora_arribo"] = df["arribo"].dt.hour.astype("int8")
df["dia_semana"] = df["retiro"].dt.day_of_week.astype("int8")

df["es_habil"] = (df["dia_semana"] < 5) & ~df["fecha"].isin(FESTIVOS) & ~df["fecha"].isin(DIAS_IRREGULARES)
df["en_horario"] = df["hora_retiro"].between(5, 23)

df["retiro_subred"] = df[COL_RETIRO].isin(SUBRED)
df["arribo_subred"] = df[COL_ARRIBO].isin(SUBRED)
df["toca_subred"] = df["retiro_subred"] | df["arribo_subred"]
df["grupo_retiro"] = df[COL_RETIRO].astype(str).map(SUBRED).fillna("Fuera").astype("category")
df["grupo_arribo"] = df[COL_ARRIBO].astype(str).map(SUBRED).fillna("Fuera").astype("category")

capacidad = estaciones.set_index("short_name")["capacity"]
df["docks_retiro"] = df[COL_RETIRO].astype(str).map(capacidad).astype("int16")
df["docks_arribo"] = df[COL_ARRIBO].astype(str).map(capacidad).astype("int16")

# Resumen por mes: insumo de los factores de escenario
habil = df.es_habil & df.en_horario
resumen_meses = pd.DataFrame({
    "viajes_limpios": df.groupby("mes", observed=True).size(),
    "dias_habiles": df[df.es_habil].groupby("mes", observed=True)["fecha"].nunique(),
    "retiros_subred": df[habil & df.retiro_subred].groupby("mes", observed=True).size(),
    "arribos_subred": df[habil & df.arribo_subred].groupby("mes", observed=True).size(),
})
resumen_meses["retiros_subred_dia"] = (resumen_meses.retiros_subred / resumen_meses.dias_habiles).round(1)
resumen_meses["arribos_subred_dia"] = (resumen_meses.arribos_subred / resumen_meses.dias_habiles).round(1)
resumen_meses

,viajes_limpios,dias_habiles,retiros_subred,arribos_subred,retiros_subred_dia,arribos_subred_dia
mes,,,,,,
2025-09,1583015,21,30072,41040,1432.0,1954.3
2025-10,1750804,23,33398,47166,1452.1,2050.7
2025-11,1596945,19,27536,39126,1449.3,2059.3
2025-12,1442200,22,24416,32007,1109.8,1454.9
2026-01,1522706,21,25326,34303,1206.0,1633.5
2026-02,1525633,19,27434,38199,1443.9,2010.5
2026-03,1650372,21,29990,41049,1428.1,1954.7
2026-04,1591433,22,28914,40628,1314.3,1846.7
2026-05,1608553,20,26023,36989,1301.2,1849.4


## 7. Guardar

Se guardan dos archivos, ambos con las mismas columnas y las mismas reglas aplicadas:

- `trips_12m.parquet`: los 12 meses, todas las estaciones. Para comparar meses, factores de escenario y movimientos de camión.
- `trips_2026-09.parquet`: solo el mes objetivo. Es el que alimenta las tasas del modelo.

Se conservan todas las estaciones porque para detectar los movimientos de camión hay que seguir cada bici aunque salga de la subred.

In [24]:
# requiere: pip install pyarrow
DIR_LIMPIO.mkdir(parents=True, exist_ok=True)
DIR_TABLAS.mkdir(parents=True, exist_ok=True)

df.to_parquet(DIR_LIMPIO / "trips_12m.parquet", index=False)
df[df.mes == MES_PRINCIPAL].to_parquet(DIR_LIMPIO / f"trips_{MES_PRINCIPAL}.parquet", index=False)

log_limpieza.to_csv(DIR_TABLAS / "log_limpieza.csv", index=False)
control.to_csv(DIR_TABLAS / "control_archivos.csv")
resumen_meses.to_csv(DIR_TABLAS / "resumen_meses.csv")

obj = pd.read_parquet(DIR_LIMPIO / f"trips_{MES_PRINCIPAL}.parquet")
print(f"12 meses: {len(df):,} viajes")
print(f"Objetivo {MES_PRINCIPAL}: {len(obj):,} viajes, {obj.loc[obj.es_habil, 'fecha'].nunique()} días hábiles")
print("Rango objetivo:", obj.retiro.min(), "→", obj.retiro.max())
print("Código 014 conservado:", obj[COL_RETIRO].isin(["014"]).any())

12 meses: 20,217,296 viajes
Objetivo 2026-09: 1,495,520 viajes, 19 días hábiles
Rango objetivo: 2026-09-01 00:00:04 → 2026-09-30 23:56:06
Código 014 conservado: True


**Qué vemos:** se eliminaron 43,859 de 18,678,140 viajes (0.235%).

- La regla que más quitó es *misma estación y menos de 3 min* (31,218 viajes, 0.167%).
- Le siguen *duración > 180 min* (7,614), *velocidad imposible* (2,988), *estación inexistente* (1,908) y *retiro fuera de la ventana* (131).
- No hubo fechas ilegibles, duplicados ni duraciones ≤ 0.

Lo eliminado es muy poco, así que la limpieza no cambia la forma de los datos.

## 8. Resumen

- **Ventana:** octubre 2025 – septiembre 2026. **Mes objetivo:** septiembre 2026 (19 días hábiles).
- **Limpieza:** se eliminó el 0.235% de los viajes (detalle en `log_limpieza`).
- **Subred:** 5 estaciones (3 Reforma, 2 Buenavista).
- **Archivos:** `trips_12m.parquet` (todo) y `trips_2026-09.parquet` (objetivo), con las mismas columnas.